<a href="https://colab.research.google.com/github/maruson08/new-folder-3/blob/main/OpenLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Claude(Sonnet 5 Medium, Colab version)

In [ ]:
# ===== 1. 필요한 라이브러리 불러오기 =====
import cv2                      # OpenCV 라이브러리
import numpy as np              # 이미지 배열 처리를 위한 numpy
from IPython.display import display, Javascript, Image  # Colab에서 JS 실행 및 이미지 출력용
from google.colab.output import eval_js   # 브라우저 JS 코드를 실행하고 결과를 받아오는 함수
from base64 import b64decode, b64encode   # 이미지 데이터를 base64로 주고받기 위한 인코딩/디코딩

# ===== 2. Haar Cascade 얼굴 인식 모델 준비 =====
# Colab에도 cv2.data.haarcascades 경로에 기본 제공 모델이 내장되어 있음
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# ===== 3. 브라우저 웹캠에서 한 장의 사진을 찍어 base64 문자열로 반환하는 JS 함수 =====
def take_photo_js(quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            // 버튼을 눌러야 캡처하도록 구성 (ESC 키 대신 '캡처 종료' 버튼 사용)
            const capturedBtn = document.createElement('button');
            capturedBtn.textContent = '사진 촬영';
            div.appendChild(video);
            div.appendChild(capturedBtn);
            document.body.appendChild(div);
            video.srcObject = stream;
            await video.play();

            // 영상 크기에 맞춰 캔버스 크기 설정
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            // 버튼 클릭을 기다림
            await new Promise((resolve) => capturedBtn.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
        takePhoto(%f)
    ''' % quality)
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    return data

# ===== 4. base64 문자열을 OpenCV 이미지(numpy 배열)로 변환하는 함수 =====
def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])  # base64 헤더 제거 후 디코딩
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)  # 바이트를 numpy 배열로 변환
    img = cv2.imdecode(jpg_as_np, flags=cv2.IMREAD_COLOR)   # numpy 배열을 이미지로 디코딩
    return img

# ===== 5. 반복적으로 사진을 찍고 얼굴을 인식하는 메인 루프 =====
print("웹캠 권한을 허용한 뒤 '사진 촬영' 버튼을 눌러 얼굴을 인식하세요.")
print("종료하려면 아래 입력창에 'q'를 입력하세요.")

while True:
    # 웹캠에서 사진 한 장 캡처 (JS 함수 호출)
    photo_data = take_photo_js()

    # base64 데이터를 OpenCV 이미지로 변환
    frame = js_to_image(photo_data)

    # 얼굴 인식을 위해 그레이스케일로 변환
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 얼굴 위치 탐지
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
    )

    # 탐지된 얼굴마다 파란색 사각형 그리기 (BGR: 255,0,0)
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)

    # 얼굴 개수를 화면 좌측 상단에 텍스트로 표시
    face_count_text = f"Faces: {len(faces)}"
    cv2.putText(
        frame, face_count_text, (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2
    )

    # 결과 이미지를 jpg로 인코딩 후 base64로 변환하여 Colab 화면에 출력
    _, buffer = cv2.imencode('.jpg', frame)
    display(Image(data=buffer.tobytes()))

    # 종료 여부를 입력받음 (ESC 키 대신 텍스트 입력으로 대체)
    user_input = input("계속하려면 Enter, 종료하려면 'q' 입력: ")
    if user_input.strip().lower() == 'q':
        print("프로그램을 종료합니다.")
        break

웹캠 권한을 허용한 뒤 '사진 촬영' 버튼을 눌러 얼굴을 인식하세요.
종료하려면 아래 입력창에 'q'를 입력하세요.


<IPython.core.display.Javascript object>

## Ver 2

In [ ]:
import time

def run_face_detection(max_frames=300, delay=0.05):
    start_stream()
    frame_count = 0

    try:
        while frame_count < max_frames:  # 무한루프 대신 최대 프레임 수 제한
            stopped = eval_js('window.streamStop')
            if stopped:
                print("종료 버튼이 눌려 스트림을 종료합니다.")
                break

            data_url = eval_js('captureFrame()')
            frame = js_to_cv2(data_url)

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(
                gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
            )

            for (x, y, w, h) in faces:
                cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)

            cv2.putText(
                frame, f"Faces: {len(faces)}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2
            )

            eval_js('showFrame("{}")'.format(cv2_to_js(frame)))

            frame_count += 1
            time.sleep(delay)  # 프레임 사이에 짧은 대기 시간을 줘서 과부하 방지

    except Exception as e:
        print(f"오류 발생: {e}")

run_face_detection()

# Claude(Sonnet 5 Medium)

In [ ]:
import cv2  # OpenCV 라이브러리 불러오기

# Haar Cascade 얼굴 인식 모델 경로 지정 및 분류기 생성
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# 웹캠 실행 (0번 카메라 사용)
cap = cv2.VideoCapture(0)

# 웹캠이 정상적으로 열렸는지 확인
if not cap.isOpened():
    print("웹캠을 열 수 없습니다.")
    exit()

while True:
    # 웹캠에서 한 프레임씩 읽어오기 (ret: 성공 여부, frame: 이미지 데이터)
    ret, frame = cap.read()
    if not ret:
        print("프레임을 읽을 수 없습니다.")
        break

    # 얼굴 인식은 흑백 이미지에서 더 빠르고 정확하게 동작하므로 그레이스케일로 변환
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 그레이스케일 이미지에서 얼굴 위치들을 탐지
    # scaleFactor: 이미지 축소 비율, minNeighbors: 얼굴로 판단할 최소 이웃 사각형 수
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
    )

    # 탐지된 얼굴마다 파란색 사각형 그리기
    for (x, y, w, h) in faces:
        # (255, 0, 0) = BGR 순서로 파란색, 두께는 2
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)

    # 인식된 얼굴 개수를 화면 좌측 상단에 텍스트로 표시
    face_count_text = f"Faces: {len(faces)}"
    cv2.putText(
        frame, face_count_text, (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2
    )

    # 결과 화면을 창에 출력
    cv2.imshow("Face Detection", frame)

    # 키 입력을 1ms 동안 대기하고, ESC(27) 입력 시 반복문 종료
    if cv2.waitKey(1) == 27:
        break

# 웹캠 자원 해제
cap.release()

# 열려있는 모든 OpenCV 창 닫기
cv2.destroyAllWindows()

# Chat GPT Incognito

In [ ]:
import cv2  # OpenCV 라이브러리를 가져옴

# Haar Cascade 얼굴 검출기 XML 파일 경로를 설정함
cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"

# 얼굴 검출기를 생성함
face_cascade = cv2.CascadeClassifier(cascade_path)

# 기본 웹캠(0번 카메라)을 열음
cap = cv2.VideoCapture(0)

# 웹캠이 정상적으로 열렸는지 확인함
if not cap.isOpened():
    # 웹캠을 열 수 없으면 오류 메시지를 출력함
    print("웹캠을 열 수 없습니다.")
    # 프로그램을 종료함
    exit()

# 웹캠이 열려 있는 동안 반복 실행함
while True:
    # 웹캠으로부터 한 프레임을 읽어옴
    ret, frame = cap.read()

    # 프레임을 정상적으로 읽지 못한 경우 반복을 종료함
    if not ret:
        break

    # 얼굴 검출 성능 향상을 위해 현재 프레임을 그레이스케일로 변환함
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 그레이스케일 영상에서 얼굴을 검출함
    faces = face_cascade.detectMultiScale(
        gray,              # 얼굴을 검출할 입력 영상
        scaleFactor=1.1,   # 이미지 축소 비율
        minNeighbors=5,    # 얼굴로 인정하기 위한 최소 이웃 개수
        minSize=(30, 30)   # 검출할 최소 얼굴 크기
    )

    # 검출된 모든 얼굴에 대해 반복함
    for (x, y, w, h) in faces:
        # 얼굴 위치에 파란색(BGR: 255, 0, 0) 사각형을 그림
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)

    # 검출된 얼굴 개수를 문자열로 생성함
    text = f"Faces: {len(faces)}"

    # 화면 좌측 상단에 얼굴 개수를 표시함
    cv2.putText(
        frame,                 # 텍스트를 표시할 영상
        text,                  # 표시할 문자열
        (10, 30),              # 텍스트 시작 좌표
        cv2.FONT_HERSHEY_SIMPLEX,  # 사용할 글꼴
        1,                     # 글자 크기
        (255, 0, 0),           # 파란색(BGR) 글자 색상
        2                      # 글자 두께
    )

    # 결과 영상을 화면에 출력함
    cv2.imshow("Real-Time Face Detection", frame)

    # 1ms 동안 키 입력을 기다리고 ESC(27)가 눌리면 반복을 종료함
    if cv2.waitKey(1) & 0xFF == 27:
        break

# 웹캠 자원을 해제함
cap.release()

# 모든 OpenCV 창을 닫음
cv2.destroyAllWindows()

# Gemini Flash Extended

In [ ]:
import cv2  # OpenCV 라이브러리를 가져옵니다.

# 1. OpenCV에 내장된 Haar Cascade 기본 얼굴 인식 분류기를 경로를 지정하여 로드합니다.
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# 2. 0번 인덱스의 기본 웹캠 장치를 활성화하여 영상 입력을 준비합니다.
cap = cv2.VideoCapture(0)

# 웹캠이 정상적으로 열려 있는 동안 실시간으로 프레임을 처리하는 루프를 시작합니다.
while cap.isOpened():
    ret, frame = cap.read()  # 웹캠으로부터 한 프레임을 읽어옵니다. (ret: 성공 여부, frame: 이미지 데이터)

    if not ret:  # 프레임을 정상적으로 읽어오지 못했다면(웹캠 연결 끊김 등) 루프를 빠져나갑니다.
        break

    # 3. Haar Cascade 연산 속도를 높이기 위해 입력받은 컬러 프레임을 흑백(Gray) 이미지로 변환합니다.
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 4. 흑백 이미지에서 얼굴을 검출합니다. (scaleFactor와 minNeighbors로 검출 예민도를 조절합니다.)
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
    )

    # 5. 검출된 얼굴들의 좌표 정보를 순회하며 사각형을 그립니다.
    for x, y, w, h in faces:
        # BGR 기준 파란색(255, 0, 0), 두께 2로 인식된 얼굴 영역에 사각형을 그립니다.
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)

    # 6. 검출된 얼굴 개수(len(faces))를 포맷팅하여 화면 좌측 상단(10, 30) 위치에 파란색 텍스트로 표시합니다.
    cv2.putText(
        frame,
        f"Faces: {len(faces)}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 0, 0),
        2,
    )

    # 7. 사각형과 텍스트가 렌더링된 최종 프레임을 화면 창('Webcam Face Detection')에 표시합니다.
    cv2.imshow("Webcam Face Detection", frame)

    # 8. 1ms 동안 키 입력을 대기하며, 사용자가 누른 키가 ESC 키(ASCII 코드 27)라면 루프를 종료합니다.
    if cv2.waitKey(1) == 27:
        break

cap.release()  # 9. 루프가 끝나면 사용 중이던 웹캠 자원을 시스템에 반환하고 해제합니다.
cv2.destroyAllWindows()  # 10. 프로그램이 완전히 종료되기 전 열려 있는 모든 OpenCV 윈도우 창을 닫습니다.